> **ARCHIVE — superseded.** This notebook is kept as a record of the research path, not as working code. Credentials were removed and outputs cleared. Results in it may rest on defects documented in `docs/RESEARCH_LOG.md` (stage: v2, point-in-time on survivors, with costs). Comments are partly in Vietnamese. The maintained implementation is `src/memescan/`.

# Meme Scanner v2 - Lean pipeline, full reasoning

## Muc tieu
Giu thuat toan v1 (da chung minh pick duoc ten sap chay: RGTI +143%, AMC +141%, WOLF +131%)
nhung loai bo MOI nguon bias da phat hien, va nang cap du lieu len Alpaca Algo Trader Plus
(SIP, 9+ nam lich su, 10,000 calls/phut).

## Nguyen tac bat di bat dich
1. Tai ngay t, moi quyet dinh chi dung du lieu den t-1 (universe) hoac den close t (trigger).
   Vao lenh tai open t+1. Stop khop theo low hoac gap trong phien.
2. Moi bien the trong cung mot bang so sanh phai chiu CUNG bo loc tradability.
   Bai hoc: v1 goc cho ro hindsight giao dich penny khong loc, con ro PIT bi loc,
   nen chenh lech $632k so voi $154k co mot nua la do loc khong deu, khong phai do ro.
3. Do IC (information coefficient) truoc, backtest sau. Backtest duoi 300 lenh voi
   phan phoi duoi day khong phan biet duoc gi - control ngau nhien tung thang ca bang.
4. Moi bang so sanh phai co TRAN (hindsight, biet la an gian) va SAN (ngau nhien).
   Ket qua chi co y nghia khi nam tren san mot cach on dinh.

## Bang loi da gap va cach phong trong notebook nay
| Loi da gap | Hau qua | Cach phong o day |
|---|---|---|
| Ro hindsight chon nam 2026 backtest tu 2021 | Thoi phong +2.8% moi lenh | Ro point-in-time, cap nhat moi ngay |
| Timestamp 05:00 so voi 00:00 khi join nguon ngoai | Social bi zero am tham, 0 ngay khop | align theo ngay lich + canh bao khi phu song = 0 |
| Extractor upper() ca cau | 46% mention la tu tieng Anh | blocklist theo do bung no (burst) |
| Dau cua feature dat theo truc giac | Social level cong duong trong khi IC am | dau dat theo IC do duoc |
| Loc thanh khoan ap khong deu giua cac ro | So sanh sai ban chat | mot ham tradability duy nhat cho moi bien the |
| Quy no-meme 3 ngay nghe hop ly | Cat mat lenh dang tich luy, ton $40k | thay bang early-exit MFE 3% sau 2 ngay (ablation ung ho) |
| Entry tai close cung gia quyet dinh | Lac quan 38% expectancy | vao open t+1, stop fill low/gap |
| ATH tinh tu 2021 | Dinh cut, sai nghia "roi tu dinh" | lich su tu 2016 |

## Nguon du lieu (citation)
| Du lieu | Nguon | Pham vi | Do tre cong bo |
|---|---|---|---|
| Gia va volume daily, SIP | Alpaca Market Data, goi Algo Trader Plus | 2016-01 den nay, split-adjusted | real-time |
| Danh sach ma tradable | Alpaca Trading API /v2/assets | active, NYSE NASDAQ AMEX ARCA BATS | hien tai |
| Short interest | FINRA bi-monthly CSV (cdn.finra.org) | 2021-01 den nay | cong 12 ngay lich |
| Reddit mentions | Arctic Shift archive (r/wallstreetbets) | 2021-01 den 2026-06, 195 ma pool cu | tre 1 ngay khi dung |
| OTM call volume | Alpaca OPRA options bars | 2024-01-22 den nay, 110 ma pool cu | tre 1 ngay khi dung |

Ghi chu quan trong: assets list chi co ma DANG active, nen universe moi van thieu ma da huy
niem yet (survivorship). Muc do lech nay lam ket qua LAC QUAN hon thuc te va duoc ghi ro
o phan ket luan. Social va options hien chi phu pool cu 195 ma nen KHONG dua vao run chinh
cua universe moi - liet ke o phan "du lieu con thieu".

## Pipeline
Hai lop, dung y tuong goc nhung dinh nghia point-in-time:

Lop 1 - Universe (tinh lai moi ngay, chi dung du lieu den t-1):
- A: gia dong cua ngay truoc duoi $10. Ly do: meme la co gia thap; IC cua "gia thap"
  duong (+0.036 toan pool, +0.061 co dieu kien).
- B: dang duoi 50% dinh moi thoi dai, voi dinh tinh tu 2016. Ly do: tai tao dung ro
  hindsight 82 ma (screen nay bat 82/82) ma khong can biet tuong lai.
- C: da tung co dot meme ket thuc truoc t (dot = chay 50% trong 20 phien; chi "biet"
  tu ngay dinh tro di, nen moc la peak + 1). Ly do: cong dong co dong cu con do,
  co che squeeze tai kich hoat khi co moi lua (backtest: +1.30% so voi +1.00%).
- Tradability (ap cho MOI bien the): ADV20 >= $1M va gia >= $0.20. Ly do: lenh $10k
  tren ma ADV duoi $1M tu day gia; duoi $0.20 spread phan tram qua lon.

Lop 2 - Trigger ignition (quyet dinh tai close t, khong doi):
RVOL >= 2, close > typical price, close > close hom truoc, close > MA20,
momentum 5 phien > 5%, va gia chua vuot MA20 qua 50% (exhaustion cap - ablation cho thay
cap 0.50 la diem ngot: chat hon thi cat mat winner bung no, long hon thi mua dinh).

Execution:
- Vao open t+1. Stop cung -8%: fill tai open neu gap qua, tai muc stop neu low cham.
- Early exit: sau 2 phien ma chua tung chay 3% thi thoat (ablation: +$21k, giu nguyen winner).
- ATR trail 3x, gay MA10 sau khi co lai, tran 60 phien.
- Chi phi giao dich: mo hinh square-root impact, one-way = 0.10% + 0.1 x sqrt(alloc/ADV20),
  tran 5%. Bao cao ca gross va net de thay chien luoc song sot sau phi hay khong.
- Danh muc: toi da 10 vi the, $10k moi lenh, khong compound.

In [ ]:
# Cell 1 - Load du lieu universe lon
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

C = Path("meme_cache"); ALLOC, CAP, MAXPOS = 10000.0, 100000.0, 10
raw = pd.read_parquet(C/"universe_v2_daily.parquet")
raw["date"] = pd.to_datetime(raw["date"])
print(f"Raw: {len(raw):,} bars | {raw.ticker.nunique():,} ma | {raw.date.min().date()} -> {raw.date.max().date()}")

# Toi uu bo nho: bo ma chua bao gio dat ADV $1M (chi la toi uu tinh toan;
# bo loc per-day van ap dung ben duoi nen ket qua khong doi)
raw["dv"] = raw["close"].astype("float64")*raw["vol"]
adv_max = raw.groupby("ticker")["dv"].apply(lambda s: s.rolling(20).mean().max())
keep = set(adv_max[adv_max >= 1e6].index)
raw = raw[raw.ticker.isin(keep)]
print(f"Sau khi bo ma khong bao gio du thanh khoan: {raw.ticker.nunique():,} ma | {len(raw):,} bars")

cl = raw.pivot_table(index="date", columns="ticker", values="close").astype("float32")
op = raw.pivot_table(index="date", columns="ticker", values="open").astype("float32")
hi = raw.pivot_table(index="date", columns="ticker", values="high").astype("float32")
lo = raw.pivot_table(index="date", columns="ticker", values="low").astype("float32")
vo = raw.pivot_table(index="date", columns="ticker", values="vol").astype("float64")
del raw
DATES = list(cl.index); TICKS = list(cl.columns)
seed82 = set(pd.read_parquet(C/"bt_daily.parquet").ticker.unique()) & set(TICKS)
print(f"Panel: {len(DATES)} phien x {len(TICKS)} ma | seed hindsight doi chieu: {len(seed82)} ma")

In [ ]:
# Cell 2 - Lop 1: universe point-in-time
cl_s = cl.shift(1)
dv20 = (cl.astype("float64")*vo).rolling(20).mean().shift(1)     # ADV den t-1
# LOI DA SUA: bo loc theo MUC gia. Alpaca adjustment=split dieu chinh NGUOC ve qua khu,
# nen ma reverse-split 1:100 nam 2023 hien gia 2021 gap 100 lan (XTIA tung hien $2.98 nghin ty).
# Loc "gia < $10" vi the dung thong tin TUONG LAI. Chi dung dac trung TY LE (split-invariant).
TRD  = (dv20 >= 1e6)                                              # tradability, ap cho MOI ro
ATH = cl_s.cummax()                                               # dinh tu 2016 den t-1
B = (cl_s/ATH - 1) <= -0.50                                       # duoi 50% dinh
ma50 = cl.rolling(50).mean().shift(1)
STABLE = cl_s > ma50                                              # da NGUNG roi, khong con dang roi

# C: tung co dot meme ket thuc truoc t (biet tu peak+1)
Cflag = pd.DataFrame(False, index=DATES, columns=TICKS)
n_ev = 0
clv = {s: cl[s].values.astype("float64") for s in TICKS}
for s in TICKS:
    c = clv[s]; n = len(c); t = 0; first_pk = None
    while t < n-1:
        if np.isnan(c[t]): t += 1; continue
        w = c[t+1:min(t+21, n)]; w = w[~np.isnan(w)]
        if len(w) == 0: t += 1; continue
        if w.max()/c[t] - 1 >= 0.50:
            seg = c[t+1:min(t+21, n)]; pk = t+1+int(np.nanargmax(seg))
            if first_pk is None: first_pk = pk
            n_ev += 1; t = pk+1
        else: t += 1
    if first_pk is not None and first_pk+1 < n:
        Cflag.iloc[first_pk+1:, TICKS.index(s)] = True

L1_PIT  = ((B & STABLE) | Cflag) & TRD
L1_HIND = pd.DataFrame(False, index=DATES, columns=TICKS)
for s in seed82: L1_HIND[s] = True
L1_HIND = L1_HIND & TRD                                           # CUNG bo loc - bai hoc loc khong deu

print(f"{n_ev} dot meme lich su, {int(Cflag.any().sum())} ma tung la meme")
print("Kich thuoc ro theo nam (ma moi ngay):")
for y in sorted({d.year for d in DATES}):
    m = [i for i, d in enumerate(DATES) if d.year == y]
    print(f"  {y}: PIT {L1_PIT.iloc[m].sum(axis=1).mean():6.0f} | hindsight+loc {L1_HIND.iloc[m].sum(axis=1).mean():5.0f}")

In [ ]:
# Cell 3 - Lop 2: trigger (vectorized) + danh sach tin hieu
ma20 = cl.rolling(20).mean().shift(1)
vavg = vo.rolling(20).mean().shift(1)
rv   = vo/vavg
typ  = (hi+lo+cl)/3
mom5 = cl/cl.shift(5) - 1
ext  = cl/ma20 - 1
TRIG = ((rv >= 2.0) & (cl > typ) & (cl > cl.shift(1)) & (cl > ma20)
        & (ext <= 0.50) & (mom5 > 0.05))
print(f"Trigger bat {int(TRIG.sum().sum()):,} lan tren toan panel (truoc khi giao voi lop 1)")

def signals(L1):
    Sm = TRIG & L1
    out = defaultdict(list)
    st = Sm.stack()
    for (d, s) in st[st].index:
        r = float(rv.at[d, s])
        out[d].append((r if not np.isnan(r) else 0.0, s))
    return out

In [ ]:
# Cell 4 - Engine backtest (next-open entry, stop intraday, chi phi square-root impact)
V1 = dict(hard_stop=0.08, atr_mult=3.0, ma_exit=10, max_hold=60, early_days=2, early_mfe=0.03)
def _atr(h, l, c, w=14):
    n = len(c); t = np.zeros(n)
    for i in range(1, n): t[i] = max(h[i]-l[i], abs(h[i]-c[i-1]), abs(l[i]-c[i-1]))
    a = np.full(n, np.nan)
    for i in range(w, n): a[i] = t[i-w+1:i+1].mean()
    return a
ARR = {s: dict(c=cl[s].values.astype("float64"), o=op[s].values.astype("float64"),
               l=lo[s].values.astype("float64"),
               atr=_atr(np.nan_to_num(hi[s].values.astype("float64")),
                        np.nan_to_num(lo[s].values.astype("float64")),
                        np.nan_to_num(cl[s].values.astype("float64")))) for s in TICKS}
DIX = {d: i for i, d in enumerate(DATES)}

def cost_oneway(alloc, adv):
    if not np.isfinite(adv) or adv <= 0: return 0.05
    return min(0.05, 0.001 + 0.1*np.sqrt(alloc/adv))

def run(sig, use_cost=True):
    trades, open_until, equity = [], [], CAP
    for d in DATES:
        if d not in sig: continue
        i0 = DIX[d]
        for score, s in sorted(sig[d], reverse=True):
            open_until = [x for x in open_until if x > d]
            if len(open_until) >= MAXPOS: break
            if equity < ALLOC: break        # LOI DA SUA: dung khi het von, tranh maxDD -655%
            a = ARR[s]; c, o, l, atr = a["c"], a["o"], a["l"], a["atr"]
            i = i0; ei = i+1
            if ei >= len(c) or np.isnan(o[ei]) or o[ei] <= 0: continue
            e = o[ei]; peak = e; xi = min(ei+V1["max_hold"], len(c)-1); ex = None; why = "max_hold"
            for j in range(ei+1, xi+1):
                if np.isnan(c[j]): continue
                sp = e*(1-V1["hard_stop"])
                if o[j] <= sp: xi, ex, why = j, o[j], "stop_gap"; break
                if l[j] <= sp: xi, ex, why = j, sp, "stop"; break
                if (j-ei) >= V1["early_days"] and (np.nanmax(c[ei+1:j+1])/e - 1) < V1["early_mfe"]:
                    xi, ex, why = j, c[j], "early_exit"; break
                peak = max(peak, c[j])
                if c[j] <= peak - V1["atr_mult"]*(atr[j] if not np.isnan(atr[j]) else 0):
                    xi, ex, why = j, c[j], "atr_trail"; break
                if c[j] > e and c[j] < np.nanmean(c[max(0, j-V1["ma_exit"]+1):j+1]):
                    xi, ex, why = j, c[j], "ma_break"; break
            if ex is None or np.isnan(ex): ex = c[xi]
            gross = ex/e - 1
            cst = 2*cost_oneway(ALLOC, float(dv20.at[d, s])) if use_cost else 0.0
            equity += ALLOC*(gross-cst)
            open_until.append(DATES[xi])
            trades.append({"ticker": s, "ei": ei, "xi": xi, "entry_date": DATES[ei],
                           "exit_date": DATES[xi], "entry_px": e, "ret": gross-cst,
                           "gross": gross, "cost": cst, "days": xi-ei, "exit_reason": why})
    tk = pd.DataFrame(trades)
    if len(tk) == 0: return None, None
    tk["pnl"] = ALLOC*tk["ret"]
    acc = defaultdict(float)
    for _, t in tk.iterrows():
        c = ARR[t["ticker"]]["c"]; sh = ALLOC/t["entry_px"]
        for j in range(int(t["ei"])+1, int(t["xi"])+1):
            if not np.isnan(c[j]) and not np.isnan(c[j-1]): acc[DATES[j]] += sh*(c[j]-c[j-1])
    for _, t in tk.iterrows():                      # tru phi vao ngay thoat de equity nhat quan voi P&L net
        acc[t["exit_date"]] -= ALLOC*t["cost"]
    eq = CAP + pd.Series([acc.get(d, 0.0) for d in DATES], index=DATES).cumsum()
    return tk, eq

def stats(tk, eq):
    r = eq.pct_change().dropna(); yrs = (DATES[-1]-DATES[0]).days/365.25
    dn = r[r < 0]
    return dict(n=len(tk), win=(tk.ret > 0).mean(), exp=tk.ret.mean(), cost=tk.cost.mean(),
                cagr=(1+tk.pnl.sum()/CAP)**(1/yrs)-1,
                sharpe=r.mean()/r.std()*np.sqrt(252) if r.std() > 0 else np.nan,
                sortino=r.mean()/dn.std()*np.sqrt(252) if len(dn) and dn.std() > 0 else np.nan,
                maxdd=((eq-eq.cummax())/eq.cummax().clip(lower=1)).min(),
                final=CAP+tk.pnl.sum())
print("Engine san sang")

In [ ]:
# Cell 5 - So sanh: PIT / hindsight cung loc (tran) / ngau nhien cung loc (san)
rng = np.random.default_rng(7)
avg_k = max(10, int(L1_PIT.sum(axis=1).mean()))
RANDU = pd.DataFrame(False, index=DATES, columns=TICKS)
trd_np = TRD.values
for i, d in enumerate(DATES):
    ok = np.where(trd_np[i])[0]
    if len(ok) > 0:
        pick = rng.choice(ok, size=min(avg_k, len(ok)), replace=False)
        RANDU.iloc[i, pick] = True

runs = {}
for lbl, U in [("PIT (hop le)", L1_PIT),
               ("HINDSIGHT 82 cung loc (tran)", L1_HIND),
               ("NGAU NHIEN cung loc (san)", RANDU)]:
    tk, eq = run(signals(U), use_cost=True)
    if tk is not None: runs[lbl] = (tk, eq)
tkg, eqg = run(signals(L1_PIT), use_cost=False)
if tkg is not None: runs["PIT khong phi (tham khao)"] = (tkg, eqg)

out = pd.DataFrame({k: stats(t, e) for k, (t, e) in runs.items()}).T[
    ["n", "win", "exp", "cost", "cagr", "sharpe", "sortino", "maxdd", "final"]]
print(out.to_string(formatters={"win": "{:.1%}".format, "exp": "{:+.2%}".format,
      "cost": "{:.2%}".format, "cagr": "{:+.1%}".format, "sharpe": "{:.2f}".format,
      "sortino": "{:.2f}".format, "maxdd": "{:.1%}".format, "final": "${:,.0f}".format}))

In [ ]:
# Cell 6 - Chi tiet ban PIT: theo nam, exit reason, tap trung P&L, equity
key = "PIT (hop le)"
tk, eq = runs[key]
print("Theo nam:")
print(tk.assign(y=pd.to_datetime(tk.entry_date).dt.year).groupby("y").agg(
      n=("ret", "size"), win=("ret", lambda x: (x > 0).mean()), pnl=("pnl", "sum")).round(3).to_string())
print("\nExit reason:")
print(tk.groupby("exit_reason").agg(n=("ret", "size"), win=("ret", lambda x: (x > 0).mean()),
      avg=("ret", "mean"), pnl=("pnl", "sum")).round(3).to_string())
top10 = tk.nlargest(10, "pnl")
print(f"\nTop 10 lenh dong gop {top10.pnl.sum()/tk.pnl.sum():.0%} tong P&L")
print(top10[["ticker", "entry_date", "entry_px", "gross", "cost", "ret", "days", "exit_reason"]]
      .assign(entry_date=lambda x: pd.to_datetime(x.entry_date).dt.date).to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for k, (t, e) in runs.items():
    ls = "--" if ("tran" in k or "san" in k) else "-"
    ax[0].plot(e.index, e.values, ls, lw=1.5, label=k)
ax[0].axhline(CAP, c="gray", lw=.8); ax[0].set_yscale("log")
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3); ax[0].set_title("Equity, log scale, da tru phi")
ax[1].plot(DATES, L1_PIT.sum(axis=1).values, lw=1.2)
ax[1].set_title("Kich thuoc ro PIT theo ngay"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
tk.to_csv(C/"v2_pit_trades.csv", index=False)
print("Saved: meme_cache/v2_pit_trades.csv")

## Du lieu con thieu va viec tiep theo
1. Ma da huy niem yet: Alpaca assets chi co ma active. Cac ten ve 0 (BBBY, MULN cu)
   thieu trong universe moi. Ket qua vi the LAC QUAN hon thuc te. Nguon co delisted
   la Polygon hoac Norgate (tra phi).
2. Social cho universe moi: Reddit mentions hien chi dem cho 195 ma pool cu. Muon phu
   universe moi phai fetch lai Arctic Shift voi danh sach ma moi, va lan nay nen luu
   ca text, so tac gia rieng biet, upvotes (mot dem chay).
3. Borrow fee, utilization, cost-to-borrow real-time: khong co nguon mien phi. Day van
   la thuoc do truc tiep nhat cua squeeze. Ortex hoac Fintel giai quyet duoc.
4. Short interest moi co FINRA bi-monthly voi do tre 12 ngay, IC gan 0 vi qua cu.
   SEC fails-to-deliver (2004+) da xac nhan tai duoc nhung chua tich hop.
5. Float va pha loang tu EDGAR: short percent of float dang thieu mau so; cac dot
   phat hanh (S-3, ATM) giai thich vi sao mot so squeeze xit nhanh.
6. Options: OPRA tu 2024-01-22, moi phu 110 ma cu. OTM call level co IC +0.061,
   manh nhat trong cac tin hieu duong da do, dang refetch cho universe moi.
7. Voi key moi, intraday 15 phut SIP tu 2016 da kha dung cho toan universe, mo duong
   cho confirmation layer moi 15 phut nhu thiet ke hai lop ban dau.